In [0]:
# 1. Load Kaggle NCAA data files
from pyspark.sql.functions import col
input_path = "/Volumes/workspace/marchmadnesslandingzone/kaggle/"
teams = spark.read.option("header", True).csv(input_path + "MTeams.csv")
seeds = spark.read.option("header", True).csv(input_path + "MNCAATourneySeeds.csv")
results = spark.read.option("header", True).csv(input_path + "MNCAATourneyCompactResults.csv")
display(teams)
display(seeds)
display(results)


In [0]:
# 2. Join seeds & teams, Season_Team, extract SeedNum
from pyspark.sql.functions import concat_ws, regexp_replace

seeds = seeds.join(teams, "TeamID", "left")
seeds = seeds.withColumn("Season_Team", concat_ws("_", col("Season"), col("TeamID")))
seed_clean = regexp_replace(regexp_replace(col("Seed"), "a", ""), "b", "")
seeds = seeds.withColumn("SeedNum", seed_clean.substr(-2, 2).cast("int"))
display(seeds)


In [0]:
# 3. Map results to submission format: ids, flags, keys
from pyspark.sql.functions import when, lit
results = results.withColumn(
    "Team1Id", when(col("WTeamID") < col("LTeamID"), col("WTeamID")).otherwise(col("LTeamID"))
).withColumn(
    "Team2Id", when(col("WTeamID") < col("LTeamID"), col("LTeamID")).otherwise(col("WTeamID"))
).withColumn(
    "id", concat_ws("_", col("Season"), col("Team1Id"), col("Team2Id"))
).withColumn(
    "Team1Wins", when(col("WTeamID") < col("LTeamID"), lit(1)).otherwise(lit(0))
)
results = results.withColumn("Team1Season_Team", concat_ws("_", col("Season"), col("Team1Id")))\
                 .withColumn("Team2Season_Team", concat_ws("_", col("Season"), col("Team2Id")))
display(results)


In [0]:
# 4. Filter and select essentials for downstream join
essentials = results.filter(col("Season") >= 2002).select(
    "id", "Team1Wins", "Team1Id", "Team2Id", "Team1Season_Team", "Team2Season_Team"
)
display(essentials)


In [0]:
# Save results essentials to marchmadness.silver_kaggle Delta table
essentials.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("marchmadness.silver_kaggle")

# Display sample of saved table for verification
display(spark.table("marchmadness.silver_kaggle"))
